# 🧠 Brain Tumor MRI Image Classification
**Domain:** Medical Imaging · **Stack:** Python, TensorFlow/Keras, Streamlit

This notebook covers the full project workflow:

1. Dataset understanding & EDA
2. Preprocessing & data augmentation
3. Custom CNN (from scratch)
4. Transfer learning (MobileNetV2, ResNet50, EfficientNetB0) with fine-tuning
5. Training with `EarlyStopping` / `ModelCheckpoint`
6. Evaluation (accuracy, precision, recall, F1, confusion matrix, history plots)
7. Model comparison & selection of the best model
8. Streamlit app generation (`app.py`) + `requirements.txt` + `README.md`

> **Dataset:** [Brain Tumor MRI Dataset (Masoud Nickparvar) – Kaggle](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset) — 4 classes: `glioma`, `meningioma`, `notumor`, `pituitary`. A GPU runtime (e.g. Colab T4) is strongly recommended.

⚠️ *Educational project – not a certified medical device.*

## 0. Setup & Imports

In [ ]:
# !pip install -q tensorflow kagglehub scikit-learn seaborn matplotlib pandas pillow streamlit
import os, json, time, random, pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, optimizers
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ---------------- Configuration ----------------
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
VAL_SPLIT   = 0.15          # taken from the Training folder
EPOCHS_CNN  = 30            # custom CNN
EPOCHS_HEAD = 10            # transfer learning: train new head (base frozen)
EPOCHS_FT   = 10            # transfer learning: fine-tune top layers
FT_LAYERS   = 30            # number of top base layers to unfreeze
MODEL_DIR   = "saved_models"
os.makedirs(MODEL_DIR, exist_ok=True)

## 1. Load the Dataset

In [ ]:
# Option A (default): download from Kaggle through kagglehub (needs internet; may need Kaggle credentials)
# Option B: set DATA_DIR manually to the folder that contains `Training` and `Testing`
DATA_DIR = None   # e.g. "/content/brain-tumor-mri-dataset"

if DATA_DIR is None:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

def find_split_root(root):
    for dp, dn, _ in os.walk(root):
        if "Training" in dn and "Testing" in dn:
            return dp
    raise FileNotFoundError("Could not find 'Training' and 'Testing' folders under " + root)

ROOT      = find_split_root(DATA_DIR)
TRAIN_DIR = os.path.join(ROOT, "Training")
TEST_DIR  = os.path.join(ROOT, "Testing")
print("Train dir:", TRAIN_DIR)
print("Test  dir:", TEST_DIR)

## 2. Understand the Dataset (EDA)

In [ ]:
def count_images(directory):
    rows = []
    for cls in sorted(os.listdir(directory)):
        p = os.path.join(directory, cls)
        if os.path.isdir(p):
            rows.append((cls, len([f for f in os.listdir(p) if f.lower().endswith((".jpg", ".jpeg", ".png"))])))
    return pd.DataFrame(rows, columns=["class", "count"])

train_counts = count_images(TRAIN_DIR).assign(split="Training")
test_counts  = count_images(TEST_DIR).assign(split="Testing")
df_counts = pd.concat([train_counts, test_counts])
CLASS_NAMES = sorted(train_counts["class"].tolist())
NUM_CLASSES = len(CLASS_NAMES)
print("Classes:", CLASS_NAMES)
display(df_counts.pivot(index="class", columns="split", values="count"))

plt.figure(figsize=(8, 4))
sns.barplot(data=df_counts, x="class", y="count", hue="split")
plt.title("Class distribution"); plt.tight_layout(); plt.show()

ratio = train_counts["count"].max() / train_counts["count"].min()
print(f"Imbalance ratio (max/min) in training set: {ratio:.2f}")

In [ ]:
# Image resolution / mode consistency (sample up to 300 images per class)
sizes, modes = [], []
for cls in CLASS_NAMES:
    files = os.listdir(os.path.join(TRAIN_DIR, cls))
    for f in random.sample(files, min(300, len(files))):
        with Image.open(os.path.join(TRAIN_DIR, cls, f)) as im:
            sizes.append(im.size); modes.append(im.mode)

size_df = pd.DataFrame(sizes, columns=["width", "height"])
print("Unique sizes (sampled):", size_df.drop_duplicates().shape[0])
print(size_df.describe().loc[["min", "mean", "max"]])
print("Image modes:", pd.Series(modes).value_counts().to_dict())

In [ ]:
# Sample images per class
fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(14, 3 * NUM_CLASSES))
for i, cls in enumerate(CLASS_NAMES):
    files = random.sample(os.listdir(os.path.join(TRAIN_DIR, cls)), 5)
    for j, f in enumerate(files):
        axes[i, j].imshow(Image.open(os.path.join(TRAIN_DIR, cls, f)).convert("RGB"))
        axes[i, j].axis("off")
        if j == 0: axes[i, j].set_title(cls, loc="left", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

## 3. Data Preprocessing & Augmentation

* **Resize** all images to 224×224, **normalize** pixels to `[0, 1]` (`Rescaling(1/255)` in the input pipeline).
* A validation set is carved out of the `Training` folder; the official `Testing` folder is kept untouched for final evaluation.
* **Augmentation** (rotation, flips, zoom, shifts, brightness) is implemented as Keras layers that are only active during training.
* **Class weights** are computed to counter mild class imbalance.

In [ ]:
common = dict(labels="inferred", label_mode="categorical", class_names=CLASS_NAMES,
              image_size=IMG_SIZE, batch_size=BATCH_SIZE, color_mode="rgb")

train_raw = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=VAL_SPLIT, subset="training",   seed=SEED, shuffle=True, **common)
val_raw   = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=VAL_SPLIT, subset="validation", seed=SEED, shuffle=False, **common)
test_raw  = keras.utils.image_dataset_from_directory(TEST_DIR, shuffle=False, **common)

normalize = layers.Rescaling(1.0 / 255)          # -> [0, 1]
AUTOTUNE = tf.data.AUTOTUNE
def prep(ds, shuffle=False):
    ds = ds.map(lambda x, y: (normalize(x), y), num_parallel_calls=AUTOTUNE)
    if shuffle: ds = ds.shuffle(1000, seed=SEED)
    return ds.prefetch(AUTOTUNE)

train_ds = prep(train_raw, shuffle=True)
val_ds   = prep(val_raw)
test_ds  = prep(test_raw)

# Class weights (from the training folder)
y_all = np.concatenate([np.full(n, i) for i, n in enumerate(train_counts.set_index("class").loc[CLASS_NAMES, "count"])])
cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_all)
CLASS_WEIGHTS = {i: w for i, w in enumerate(cw)}
print("Class weights:", {CLASS_NAMES[i]: round(w, 3) for i, w in CLASS_WEIGHTS.items()})

In [ ]:
# Augmentation block (active only when training=True)
def build_augmentation():
    return keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.15),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomBrightness(0.2, value_range=(0, 1)),
    ], name="augmentation")

# Visualize augmentation
aug = build_augmentation()
img_batch, _ = next(iter(train_ds))
plt.figure(figsize=(12, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    out = aug(tf.expand_dims(img_batch[0], 0), training=True)[0]
    plt.imshow(tf.clip_by_value(out, 0, 1)); plt.axis("off")
plt.suptitle("Augmented versions of the same MRI image"); plt.tight_layout(); plt.show()

## 4. Shared Training Utilities
Callbacks (`EarlyStopping`, `ModelCheckpoint`, `ReduceLROnPlateau`), a training helper, and an evaluation helper.

In [ ]:
def get_callbacks(name, patience=6):
    ckpt = os.path.join(MODEL_DIR, f"{name}.h5")
    return [
        callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True, verbose=1),
        callbacks.ModelCheckpoint(ckpt, monitor="val_loss", save_best_only=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ]

def merge_histories(*hists):
    merged = {}
    for h in hists:
        for k, v in h.history.items():
            merged.setdefault(k, []).extend(v)
    return merged

def plot_history(hist, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(hist["accuracy"], label="train"); ax[0].plot(hist["val_accuracy"], label="val")
    ax[0].set_title(f"{title} – Accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(hist["loss"], label="train"); ax[1].plot(hist["val_loss"], label="val")
    ax[1].set_title(f"{title} – Loss"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()

RESULTS = {}     # name -> dict of metrics
HISTORIES = {}   # name -> history dict

def evaluate_model(model, name):
    y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds])
    t0 = time.time()
    probs = model.predict(test_ds, verbose=0)
    infer_ms = (time.time() - t0) / len(y_true) * 1000
    y_pred = np.argmax(probs, axis=1)

    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro")
    print(f"\n===== {name} : TEST RESULTS =====")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5.5, 4.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f"{name} – Confusion Matrix"); plt.ylabel("True"); plt.xlabel("Predicted")
    plt.tight_layout(); plt.show()

    RESULTS[name] = dict(Accuracy=acc, Precision=p, Recall=r, F1=f1,
                         Params_M=model.count_params() / 1e6, Infer_ms_per_img=infer_ms)
    return RESULTS[name]

## 5. Model A – Custom CNN (from scratch)
Four convolutional blocks (Conv → BatchNorm → Conv → BatchNorm → MaxPool → Dropout) followed by a dense head with dropout.

In [ ]:
def conv_block(x, filters, drop):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    return layers.Dropout(drop)(x)

def build_custom_cnn():
    inp = layers.Input(shape=(*IMG_SIZE, 3))          # input already scaled to [0, 1]
    x = build_augmentation()(inp)
    for f, d in [(32, 0.1), (64, 0.15), (128, 0.2), (256, 0.25)]:
        x = conv_block(x, f, d)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return models.Model(inp, out, name="custom_cnn")

cnn = build_custom_cnn()
cnn.compile(optimizer=optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
cnn.summary()

In [ ]:
h = cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_CNN,
            class_weight=CLASS_WEIGHTS, callbacks=get_callbacks("custom_cnn", patience=8))
HISTORIES["Custom CNN"] = merge_histories(h)
plot_history(HISTORIES["Custom CNN"], "Custom CNN")
evaluate_model(cnn, "Custom CNN");

## 6. Transfer Learning
Pretrained ImageNet backbones (`MobileNetV2`, `ResNet50`, `EfficientNetB0`) with a new classification head.

* **Phase 1:** freeze the backbone, train only the new head.
* **Phase 2:** unfreeze the top `FT_LAYERS` layers (BatchNorm layers stay frozen) and fine-tune with a very small learning rate.

Because our pipeline normalizes to `[0, 1]`, a small `Rescaling` / `Normalization` adapter inside each model converts inputs to the range that backbone expects (these are standard layers, so the models save cleanly to `.h5`).

In [ ]:
from tensorflow.keras.applications import MobileNetV2, ResNet50, EfficientNetB0

def preprocess_adapter(kind):
    if kind == "mobilenet":      # expects [-1, 1]
        return layers.Rescaling(2.0, offset=-1.0, name="to_minus1_1")
    if kind == "efficientnet":   # expects [0, 255] (rescaling is built into the model)
        return layers.Rescaling(255.0, name="to_0_255")
    if kind == "resnet":         # [0,255] minus ImageNet RGB mean
        return keras.Sequential([
            layers.Rescaling(255.0),
            layers.Normalization(mean=[123.68, 116.779, 103.939], variance=[1.0, 1.0, 1.0])
        ], name="resnet_preprocess")

BACKBONES = {
    "MobileNetV2":    (MobileNetV2,    "mobilenet"),
    "ResNet50":       (ResNet50,       "resnet"),
    "EfficientNetB0": (EfficientNetB0, "efficientnet"),
}

def build_transfer_model(name):
    ctor, kind = BACKBONES[name]
    base = ctor(include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3))
    base.trainable = False
    inp = layers.Input(shape=(*IMG_SIZE, 3))
    x = build_augmentation()(inp)
    x = preprocess_adapter(kind)(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return models.Model(inp, out, name=name.lower()), base

def train_transfer(name):
    print(f"\n{'='*20} {name} {'='*20}")
    model, base = build_transfer_model(name)
    ckpt_name = name.lower()

    # Phase 1 – train the head
    model.compile(optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
                   class_weight=CLASS_WEIGHTS, callbacks=get_callbacks(ckpt_name, patience=4))

    # Phase 2 – fine-tune top layers
    base.trainable = True
    for layer in base.layers[:-FT_LAYERS]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    model.compile(optimizers.Adam(1e-5), loss="categorical_crossentropy", metrics=["accuracy"])
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT,
                   class_weight=CLASS_WEIGHTS, callbacks=get_callbacks(ckpt_name + "_ft", patience=4))

    HISTORIES[name] = merge_histories(h1, h2)
    plot_history(HISTORIES[name], name)
    evaluate_model(model, name)
    return model

trained = {}
for name in BACKBONES:
    trained[name] = train_transfer(name)

## 7. Model Comparison

In [ ]:
results_df = pd.DataFrame(RESULTS).T.sort_values("F1", ascending=False)
display(results_df.style.format({"Accuracy": "{:.4f}", "Precision": "{:.4f}", "Recall": "{:.4f}",
                                 "F1": "{:.4f}", "Params_M": "{:.2f}", "Infer_ms_per_img": "{:.2f}"})
        .background_gradient(cmap="Greens", subset=["Accuracy", "F1"]))

ax = results_df[["Accuracy", "Precision", "Recall", "F1"]].plot(kind="bar", figsize=(10, 4.5), rot=15)
ax.set_ylim(max(0, results_df[["Accuracy", "Precision", "Recall", "F1"]].min().min() - 0.05), 1.0)
ax.set_title("Model comparison on the held-out test set"); plt.tight_layout(); plt.show()

# Validation curves overlay
plt.figure(figsize=(10, 4))
for n, hh in HISTORIES.items():
    plt.plot(hh["val_accuracy"], label=n)
plt.title("Validation accuracy across epochs"); plt.xlabel("epoch"); plt.legend(); plt.show()

In [ ]:
# Pick the best model: highest F1 (macro); ties broken by smaller size / faster inference
best_name = results_df.sort_values(["F1", "Params_M"], ascending=[False, True]).index[0]
print("🏆 Best model:", best_name)
print(results_df.loc[best_name])

best_model = cnn if best_name == "Custom CNN" else trained[best_name]
best_model.save("best_model.h5")
with open("class_names.json", "w") as f:
    json.dump(CLASS_NAMES, f)
results_df.to_csv("model_comparison.csv")
print("Saved: best_model.h5, class_names.json, model_comparison.csv")

**Analysis guidance (fill in after running):**
* *Most accurate:* the row with the highest Accuracy/F1 above.
* *Most efficient:* compare `Params_M` and `Infer_ms_per_img` (MobileNetV2 is usually the lightest).
* *Most reliable:* look at per-class recall in the classification reports — in medical imaging, missing a tumor (false negatives) is costlier than a false alarm.

## 8. Sanity-check Inference on a Single Image

In [ ]:
def predict_image(model, path):
    img = Image.open(path).convert("RGB").resize(IMG_SIZE)
    arr = np.expand_dims(np.asarray(img, dtype="float32") / 255.0, 0)
    probs = model.predict(arr, verbose=0)[0]
    return CLASS_NAMES[int(np.argmax(probs))], probs

sample_cls = random.choice(CLASS_NAMES)
sample_path = os.path.join(TEST_DIR, sample_cls, random.choice(os.listdir(os.path.join(TEST_DIR, sample_cls))))
pred, probs = predict_image(best_model, sample_path)
plt.imshow(Image.open(sample_path).convert("RGB")); plt.axis("off")
plt.title(f"True: {sample_cls} | Pred: {pred} ({probs.max():.1%})"); plt.show()

## 9. Streamlit Application
The cell below writes `app.py`. The app loads `best_model.h5` + `class_names.json`, lets the user upload an MRI image, and shows the predicted tumor type with confidence scores.

In [ ]:
%%writefile app.py
import json
import numpy as np
import pandas as pd
import streamlit as st
import tensorflow as tf
from PIL import Image

IMG_SIZE = (224, 224)

st.set_page_config(page_title="Brain Tumor MRI Classifier", page_icon="🧠", layout="centered")

@st.cache_resource
def load_assets():
    model = tf.keras.models.load_model("best_model.h5", compile=False)
    with open("class_names.json") as f:
        class_names = json.load(f)
    return model, class_names

model, class_names = load_assets()

DESCRIPTIONS = {
    "glioma": "Tumor arising from glial cells in the brain or spinal cord.",
    "meningioma": "Tumor arising from the meninges, the membranes covering the brain.",
    "pituitary": "Tumor of the pituitary gland at the base of the brain.",
    "notumor": "No tumor pattern detected in this scan.",
}

st.title("🧠 Brain Tumor MRI Classifier")
st.write("Upload a brain MRI image and the model will predict the tumor type "
         "(glioma, meningioma, pituitary, or no tumor).")

with st.sidebar:
    st.header("About")
    st.write("Deep-learning model trained on the Brain Tumor MRI Multi-Class dataset.")
    st.write("Classes: " + ", ".join(class_names))
    st.warning("For educational use only. This is NOT a medical diagnosis. "
               "Always consult a qualified radiologist or physician.")

uploaded = st.file_uploader("Upload an MRI image", type=["jpg", "jpeg", "png"])

if uploaded is not None:
    image = Image.open(uploaded).convert("RGB")
    col1, col2 = st.columns(2)
    col1.image(image, caption="Uploaded MRI", use_container_width=True)

    arr = np.asarray(image.resize(IMG_SIZE), dtype="float32") / 255.0
    probs = model.predict(np.expand_dims(arr, 0), verbose=0)[0]
    idx = int(np.argmax(probs))
    label = class_names[idx]

    with col2:
        st.subheader("Prediction")
        st.metric("Tumor type", label.upper(), f"{probs[idx]*100:.2f}% confidence")
        st.caption(DESCRIPTIONS.get(label, ""))
        if probs[idx] < 0.60:
            st.warning("Low confidence - please review this scan with a specialist.")

    st.subheader("Confidence scores")
    df = pd.DataFrame({"Class": class_names, "Confidence (%)": probs * 100}).set_index("Class")
    st.bar_chart(df)
    st.dataframe(df.style.format("{:.2f}"))
else:
    st.info("Please upload an image to get a prediction.")

In [ ]:
%%writefile requirements.txt
tensorflow>=2.13
streamlit>=1.30
numpy
pandas
pillow
scikit-learn
matplotlib
seaborn

### Run the app
**Locally:**
```bash
pip install -r requirements.txt
streamlit run app.py
```
**From Google Colab** (run the next cell, then open the printed URL):

In [ ]:
# Colab only: expose the Streamlit app through a localtunnel
# !npm install -g localtunnel -q
# !streamlit run app.py --server.port 8501 &>/content/logs.txt &
# !echo "Password (your public IP):" && curl -s ipv4.icanhazip.com
# !npx localtunnel --port 8501

## 10. README for GitHub

In [ ]:
%%writefile README.md
# Brain Tumor MRI Image Classification

Deep-learning pipeline that classifies brain MRI scans into **glioma, meningioma, pituitary, or no tumor**,
with a custom CNN built from scratch and transfer-learning models (MobileNetV2, ResNet50, EfficientNetB0),
deployed through a Streamlit web app.

## Project structure
```
├── brain_tumor_mri_classification.ipynb   # EDA, training, evaluation, comparison
├── app.py                                 # Streamlit application
├── best_model.h5                          # best-performing model
├── class_names.json
├── model_comparison.csv
├── saved_models/                          # all trained models (.h5)
└── requirements.txt
```

## Workflow
1. Dataset exploration (class balance, resolution, sample images)
2. Preprocessing: resize 224x224, normalize to [0, 1]
3. Augmentation: flips, rotation, zoom, shifts, brightness
4. Custom CNN with BatchNorm + Dropout
5. Transfer learning (frozen head training, then fine-tuning top layers)
6. Callbacks: EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
7. Evaluation: accuracy, precision, recall, F1, confusion matrix, history plots
8. Model comparison and deployment of the best model

## Results
See `model_comparison.csv` (fill in your table here after running the notebook).

## Run
```bash
pip install -r requirements.txt
streamlit run app.py
```

## Dataset
[Brain Tumor MRI Dataset - Kaggle](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset)

## Disclaimer
Educational project only. Not intended for clinical use.

## 11. Conclusion & Future Work
* Summarize which model won and why (accuracy, F1, per-class recall, size, speed).
* Future work: Grad-CAM explainability, test-time augmentation, ensembling, k-fold cross-validation, external-dataset validation, ONNX/TFLite export for edge deployment.